<a href="https://colab.research.google.com/github/vencov/FAV_course/blob/main/TutCochModels/TutDRNLmodelTuningCurves_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tuning curves with DRNL model - HOMEWORK

A tuning curve here is found the classic constant-response way:

1. **Find the criterion response.** Drive the full 300-channel model with a
   reference tone (e.g. 1 kHz at 10 dB SPL) and find the BM place with the
   *maximum* response amplitude — same traveling-wave-peak search as in
   Section 1. That place's CF is the "recording site", and its response
   magnitude is the **criterion** (target) amplitude.
2. **Trace the curve.** At that same fixed site, sweep the stimulus
   *frequency* and, for each frequency, do a bisection search over stimulus
   *level* to find the level that reproduces the criterion amplitude at that
   site.
3. Plotting threshold level (dB SPL) vs. frequency gives the usual V-shaped
   tuning curve, with its tip at the reference frequency/level.

`np.nan` is returned for a test frequency if no level within `level_bounds`
reaches the criterion (i.e. the curve would run off the top of the plot) —
this is normal for the tuning-curve tail.

In [ ]:
import numpy as np
from scipy.signal import lfilter
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 5)



def erb_space(low_freq, high_freq, n=100):
    """Array of n frequencies uniformly spaced on an ERB scale between low_freq
    and high_freq (Moore & Glasberg, 1983). Port of ERBSpace.m."""
    ear_q = 9.26449   # Glasberg & Moore parameters
    min_bw = 24.7
    idx = np.arange(n, 0, -1)
    cf_array = -(ear_q * min_bw) + np.exp(
        idx * (-np.log(high_freq + ear_q * min_bw) + np.log(low_freq + ear_q * min_bw)) / n
    ) * (high_freq + ear_q * min_bw)
    return cf_array

def get_drnl_param(cf):
    """DRNL filter parameters for normal hearing (Morten Jepsen, 2007).
    Port of getDRNLparam.m. Returns (lin, nlin) parameter dicts."""
    lin, nlin = {}, {}

    lin['CF_lin'] = 10 ** (-0.06762 + 1.01679 * np.log10(cf))
    lin['nGTfilt_lin'] = 2                                   # cascaded gammatone filters
    lin['LP_lin_cutoff'] = 10 ** (-0.06762 + 1.01 * np.log10(cf))
    lin['nLPfilt_lin'] = 4                                   # cascaded lowpass filters
    lin['g'] = 10 ** (4.20405 - 0.47909 * np.log10(cf))      # linear-path gain
    lin['BW_lin'] = 10 ** (0.03728 + 0.75 * np.log10(cf))

    nlin['CF_nlin'] = 10 ** (-0.05252 + 1.01650 * np.log10(cf))
    nlin['nGTfilt_nlin'] = 2
    nlin['BW_nlin'] = 10 ** (-0.03193 + 0.7 * np.log10(cf))
    nlin['LP_nlin_cutoff'] = 10 ** (-0.05252 + 1.01 * np.log10(cf))
    nlin['nLPfilt_nlin'] = 1

    # broken-stick nonlinearity coefficients; the 1500 Hz clamp for a,b below
    # CF matches the original comment: "the 1500 assumption is no good for
    # compression at low freq filters" but is what the MATLAB code does.
    if cf <= 1000:
        nlin['a'] = 10 ** (1.40298 + 0.81916 * np.log10(cf))
        nlin['b'] = 10 ** (1.61912 - 0.81867 * np.log10(cf))
    else:
        nlin['a'] = 10 ** (1.40298 + 0.81916 * np.log10(1500))
        nlin['b'] = 10 ** (1.61912 - 0.81867 * np.log10(1500))
    nlin['c'] = 10 ** (-0.60206)   # compression exponent

    return lin, nlin

def coef_gt_drnl(fc, bw, n, fs):
    """Gammatone filter coefficients for the DRNL implementation. Port of
    coefGtDRNL.m. n cascades are pre-convolved into a single (b, a) pair
    (the run-scripts always call this with n=1 and cascade by re-filtering
    instead, see drnl() below)."""
    theta = 2 * np.pi * fc / fs
    phi = 2 * np.pi * bw / fs
    alpha = -np.exp(-phi) * np.cos(theta)

    b1 = 2 * alpha
    b2 = np.exp(-2 * phi)
    a0 = np.abs(
        (1 + b1 * np.cos(theta) - 1j * b1 * np.sin(theta)
         + b2 * np.cos(2 * theta) - 1j * b2 * np.sin(2 * theta))
        / (1 + alpha * np.cos(theta) - 1j * alpha * np.sin(theta))
    )
    a1 = alpha * a0

    b = np.array([a0, a1])
    a = np.array([1.0, b1, b2])

    bb, aa = np.array([1.0]), np.array([1.0])
    for _ in range(n):
        bb = np.convolve(b, bb)
        aa = np.convolve(a, aa)
    return bb, aa


def coef_lp_drnl(fc, fs):
    """2nd-order lowpass filter coefficients for the DRNL implementation.
    Port of coefLPDRNL.m."""
    theta = np.pi * fc / fs
    cot_theta = 1.0 / np.tan(theta)
    C = 1 / (1 + np.sqrt(2) * cot_theta + cot_theta ** 2)
    D = 2 * C * (1 - cot_theta ** 2)
    E = C * (1 - np.sqrt(2) * cot_theta + cot_theta ** 2)
    return np.array([C, 2 * C, C]), np.array([1.0, D, E])

def gaussgrid(n):
    """BM site array covering 0-1 with Gaussian-distributed point density.
    Port of gaussgrid.m (plotting option omitted; use n = number of channels)."""
    stred = 0.35
    sig1 = 2 * (1.2) ** 2
    sig2 = 2 * (0.46) ** 2

    xo = np.arange(1, n + 1) / n
    ind = int(np.ceil(stred * n))
    center = xo[ind - 1]

    dx = np.zeros(n)
    dx[:ind] = np.exp(((xo[:ind] - center) ** 2) / sig1)
    dx[ind:] = np.exp(((xo[ind:] - center) ** 2) / sig2)
    dx = dx / np.sum(dx)

    x = np.cumsum(dx)
    if x[-1] > 1:
        x[-1] = 1.0
    return x

def drnl(fs, signal, cf_array):
    """Dual-resonance nonlinear (DRNL) basilar-membrane filter
    (Lopez-Poveda & Meddis, 2001). Port of drnl.m.

    Parameters
    ----------
    fs : float
        Sampling rate (Hz).
    signal : 1-D array
        Input pressure waveform (Pa).
    cf_array : 1-D array
        Characteristic frequencies, ordered LOW to HIGH (as in the original code).

    Returns
    -------
    out : ndarray, shape (n_samples, n_channels)
        BM displacement per channel, re-ordered HIGH to LOW CF — i.e. back to
        the original `cf_new` ordering (base -> apex), same as MATLAB's
        `out = fliplr(outR)`.
    """
    cf_array = np.atleast_1d(np.asarray(cf_array, dtype=float))
    signal = np.asarray(signal, dtype=float)
    n_ch = len(cf_array)
    outR = np.zeros((len(signal), n_ch))

    for ch in range(n_ch):
        cf = cf_array[ch]
        lin, nlin = get_drnl_param(cf)

        # --- linear path ---------------------------------------------------
        GTlin_b, GTlin_a = coef_gt_drnl(lin['CF_lin'], lin['BW_lin'], 1, fs)
        LPlin_b, LPlin_a = coef_lp_drnl(lin['LP_lin_cutoff'], fs)

        y_lin = signal * lin['g']
        for _ in range(lin['nGTfilt_lin']):
            y_lin = np.real(lfilter(GTlin_b, GTlin_a, y_lin))
        for _ in range(lin['nLPfilt_lin']):
            y_lin = lfilter(LPlin_b, LPlin_a, y_lin)

        # --- nonlinear ("broken stick") path --------------------------------
        GTnlin_b, GTnlin_a = coef_gt_drnl(nlin['CF_nlin'], nlin['BW_nlin'], 1, fs)
        LPnlin_b, LPnlin_a = coef_lp_drnl(nlin['LP_nlin_cutoff'], fs)

        y_nlin = signal.copy()
        for _ in range(nlin['nGTfilt_nlin']):
            y_nlin = np.real(lfilter(GTnlin_b, GTnlin_a, y_nlin))

        a_, b_, c_ = nlin['a'], nlin['b'], nlin['c']
        y_decide = np.vstack([a_ * np.abs(y_nlin), b_ * np.abs(y_nlin) ** c_])
        y_nlin = np.sign(y_nlin) * np.min(y_decide, axis=0)   # broken-stick compression

        for _ in range(nlin['nGTfilt_nlin']):
            y_nlin = np.real(lfilter(GTnlin_b, GTnlin_a, y_nlin))
        for _ in range(nlin['nLPfilt_nlin']):
            y_nlin = lfilter(LPnlin_b, LPnlin_a, y_nlin)

        outR[:, ch] = y_nlin + y_lin

    return np.fliplr(outR)

  # Values taken from fce_DRNLjepsen/cf_nobiliEXP_R3107_10dBOME.mat ('cf_new'),
# base (high CF) -> apex (low CF), 300 channels in order to get very fine freq resolution
cf_new = np.array([
    18737.739830, 18512.897404, 18288.054979, 18063.212554, 17838.370129, 17613.527703,
    17388.685278, 17163.842853, 16939.000428, 16714.158002, 16489.315577, 16264.473152,
    15826.723294, 15613.657479, 15400.591664, 15193.180785, 14985.769906, 14783.863876,
    14581.957846, 14188.863279, 13997.532516, 13806.201753, 13619.949060, 13433.696368,
    13252.386969, 13071.077571, 12894.580267, 12718.082964, 12546.270039, 12374.457114,
    12207.204241, 12039.951367, 11877.137517, 11714.323667, 11555.831026, 11397.338385,
    11243.052265, 11088.766144, 10938.574899, 10788.383655, 10642.178605, 10495.973555,
    10401.090454, 10306.207353, 10211.324251, 10072.777008, 9934.229766, 9799.359677,
    9664.489588, 9533.199058, 9401.908528, 9274.102552, 9146.296577, 9021.882674,
    8897.468771, 8776.356911, 8655.245052, 8537.347597, 8419.450142, 8304.681779,
    8189.913417, 8078.191097, 7966.468778, 7857.711657, 7748.954537, 7643.083917,
    7537.213296, 7434.152566, 7331.091836, 7230.766420, 7130.441003, 7032.778303,
    6935.115602, 6840.044948, 6744.974293, 6652.426888, 6559.879484, 6499.818736,
    6439.757987, 6379.697238, 6291.997206, 6204.297173, 6118.924768, 6033.552364,
    5950.445810, 5867.339257, 5786.438418, 5705.537578, 5626.783912, 5548.030245,
    5471.366762, 5394.703280, 5320.074508, 5245.445735, 5172.797670, 5100.149604,
    5029.429675, 4958.709747, 4889.866781, 4821.023816, 4776.346604, 4731.669391,
    4686.992179, 4621.755013, 4556.517848, 4493.012128, 4429.506409, 4367.686181,
    4305.865954, 4245.686485, 4185.507015, 4126.924757, 4068.342498, 4011.315059,
    3954.287620, 3898.773735, 3843.259849, 3789.219346, 3735.178843, 3682.572618,
    3629.966393, 3578.756378, 3527.546364, 3494.312458, 3461.078551, 3427.844644,
    3379.316866, 3330.789087, 3283.549275, 3236.309462, 3190.323431, 3144.337401,
    3099.571876, 3054.806351, 3011.228938, 2967.651526, 2925.230692, 2882.809859,
    2841.514908, 2800.219958, 2760.021008, 2719.822057, 2680.690019, 2641.557981,
    2603.464539, 2565.371096, 2528.288684, 2491.206271, 2455.108055, 2419.009839,
    2383.869699, 2348.729558, 2314.522065, 2280.314571, 2247.014972, 2213.715372,
    2181.299571, 2148.883769, 2117.328309, 2085.772848, 2055.054895, 2024.336941,
    1994.434266, 1964.531592, 1935.422558, 1906.313523, 1877.977066, 1849.640609,
    1822.056224, 1794.471839, 1767.619565, 1740.767291, 1714.627698, 1688.488105,
    1663.042278, 1637.596450, 1612.825975, 1588.055500, 1563.942453, 1539.829406,
    1516.356339, 1492.883271, 1470.033197, 1447.183123, 1424.939509, 1402.695894,
    1359.389391, 1338.310834, 1317.232277, 1296.713161, 1276.194046, 1256.219524,
    1236.245002, 1216.800620, 1197.356238, 1178.427925, 1159.499612, 1141.073672,
    1122.647731, 1104.710830, 1086.773928, 1051.852245, 1034.854828, 1017.857410,
    1001.311118, 984.764826, 968.657685, 952.550545, 936.870901, 921.191256,
    890.664268, 875.805879, 860.947490, 846.483455, 832.019420, 817.939271,
    803.859123, 790.152673, 776.446223, 749.760882, 736.772337, 723.783792,
    711.139974, 698.496155, 686.187913, 673.879671, 649.916529, 638.252958,
    626.589387, 615.235377, 603.881366, 581.776034, 571.016715, 560.257395,
    549.783637, 539.309878, 529.114101, 518.918325, 499.067979, 489.406228,
    479.744478, 460.933837, 442.622446, 433.709750, 424.797053, 407.444761,
    398.998887, 390.553012, 374.109584, 358.102577, 350.311493, 342.520409,
    327.351805, 312.585786, 305.398728, 298.211670, 284.219054, 277.408434,
    270.597814, 257.338091, 244.430293, 238.147685, 231.865077, 219.633353,
    207.726268, 201.930737, 196.135206, 184.851781, 173.867826, 168.521610,
    163.175395, 152.766749, 142.634356, 137.702621, 132.770886, 123.169200,
    113.822350, 104.723573, 95.866284, 87.244076, 83.047391, 78.850707,
    70.680105, 66.703231, 62.726357, 54.983708, 51.215132, 47.446556,
    42.620059, 37.793563, 32.967067, 26.014253, 19.061439, 12.108626,
    5.155812, 5, 4.5, 4.4, 4.3, 4,
])


CF_full = cf_new[::-1]  # low -> high, as drnl() expects (MATLAB: fliplr(cf_new))
print(f"{len(CF_full)} channels, CF range: {CF_full[0]:.1f}-{CF_full[-1]:.1f} Hz")


def bm_response_at_channel(freq, level_db, cf, fs=40e3, scaleDRNL=5e-4,
                            Nper=40, Tonset=30e-3, Rdur=10e-3):
    """BM displacement magnitude at model place `cf`, evoked by a ramped
    pure tone of the given frequency and level."""
    Nonset = round(Tonset * fs)
    Nstat = int(round(Nper * fs / freq))
    Tdur = (Nonset + Nstat) / fs
    tx = np.arange(0, Tdur + 1 / fs, 1 / fs)
    sig = np.sqrt(2) * np.sin(2 * np.pi * freq * tx) * scaleDRNL * 10 ** (level_db / 20) * 2e-5

    rx = np.arange(0, Rdur + 1 / fs, 1 / fs)
    rx = np.pi * rx / Rdur
    rampUp = (1 + np.cos(rx + np.pi)) / 2
    wholeramp = np.concatenate([rampUp, np.ones(len(sig) - len(rx))])
    sig = sig * wholeramp

    out = drnl(fs, sig, np.array([cf]))
    seg = out[Nonset:Nonset + Nstat, 0]
    fx = np.arange(Nstat) * fs / Nstat
    idx = int(np.argmin(np.abs(fx - freq)))
    spec = np.fft.fft(seg) / Nstat
    return np.abs(spec[idx])


from scipy.optimize import brentq

def tuning_curve(ref_freq, ref_level_db, freq_range=None, n_freqs=41,
                  level_bounds=(-20, 120), fs=40e3, scaleDRNL=5e-4,
                  Nper=40, Tonset=30e-3, Rdur=10e-3, xtol=0.5):
    """Iso-response tuning curve of the DRNL model.

    Parameters
    ----------
    ref_freq, ref_level_db : reference tone (Hz, dB SPL) that defines the
        BM place (best-excited channel) and the criterion response amplitude.
    freq_range : (low, high) Hz to sweep; defaults to (ref_freq/4, ref_freq*3).
    n_freqs : number of test frequencies (log-spaced).
    level_bounds : (min, max) dB SPL searched for each frequency.
    xtol : level search tolerance, dB.

    Returns
    -------
    place_cf : CF (Hz) of the model channel used as the fixed recording site.
    target : criterion BM-displacement amplitude at that site.
    freqs, thresholds : test frequencies and the level (dB SPL, NaN if
        off-scale) needed at each to match `target`.
    """
    # --- 1. find the best-excited channel for the reference tone -----------
    Nonset = round(Tonset * fs)
    Nstat = int(fs * 30 / ref_freq)
    Tdur = (Nonset + Nstat) / fs
    tx = np.arange(0, Tdur + 1 / fs, 1 / fs)
    sig = np.sqrt(2) * np.sin(2 * np.pi * ref_freq * tx) * scaleDRNL * 10 ** (ref_level_db / 20) * 2e-5
    rx = np.arange(0, Rdur + 1 / fs, 1 / fs)
    rx = np.pi * rx / Rdur
    rampUp = (1 + np.cos(rx + np.pi)) / 2
    wholeramp = np.concatenate([rampUp, np.ones(len(sig) - len(rx))])
    sig = sig * wholeramp

    out = drnl(fs, sig, CF_full)                 # full traveling wave, as in Section 1
    seg = out[Nonset:Nonset + Nstat, :]
    fx = np.arange(Nstat) * fs / Nstat
    idxFreq = int(np.argmin(np.abs(fx - ref_freq)))
    ExcPat = np.fft.fft(seg, axis=0)[idxFreq, :] / Nstat
    idxExc = int(np.argmax(np.abs(ExcPat)))
    place_cf = CF_full[::-1][idxExc]              # channels are HIGH->LOW ordered (see §1)
    target = np.abs(ExcPat[idxExc])

    # --- 2. iso-response search at that place, across frequency ------------
    if freq_range is None:
        freq_range = (ref_freq / 4, ref_freq * 3)
    freqs = np.sort(np.unique(np.concatenate(
        [np.geomspace(freq_range[0], freq_range[1], n_freqs), [ref_freq]])))

    thresholds = np.full(len(freqs), np.nan)
    for i, f0 in enumerate(freqs):
        def g(level, f0=f0):
            return bm_response_at_channel(f0, level, place_cf, fs, scaleDRNL, Nper, Tonset, Rdur) - target
        lo, hi = level_bounds
        g_lo, g_hi = g(lo), g(hi)
        if g_lo >= 0:
            thresholds[i] = lo       # already above criterion at the lowest level tried
        elif g_hi < 0:
            thresholds[i] = np.nan   # criterion never reached -> off scale
        else:
            thresholds[i] = brentq(g, lo, hi, xtol=xtol)

    return place_cf, target, freqs, thresholds


place_cf, target, tc_freqs, tc_levels = tuning_curve(ref_freq=1000, ref_level_db=100)

plt.figure(figsize=(7, 5))
plt.semilogx(tc_freqs, tc_levels, marker='o', ms=3)
plt.scatter([1000], [10], color='red', zorder=5, label='reference point')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Threshold level (dB SPL)')
plt.title(f'DRNL iso-response tuning curve — place CF = {place_cf:.0f} Hz')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.show()


### Cell with HOMEWORK. Complete the function estimate_erb()

In [ ]:
# np.trapz was renamed np.trapezoid in newer NumPy; this works with either.
_trapz = getattr(np, 'trapezoid', None) or np.trapz

def erb_cam(cf_hz):
    """Human ERB (Hz) at center frequency cf_hz, from the Cams (ERB-rate) scale
    of Glasberg & Moore (1990):  ERB_N(f) = 24.7 * (4.37*f/1000 + 1).
    Level-independent -- this is the classic behavioral reference, not
    something derived from the model."""
    return 24.7 * (4.37 * cf_hz / 1000 + 1)


def estimate_erb(freqs, levels, fill_value='zero'):
    """Estimate the Equivalent Rectangular Bandwidth (ERB, in Hz) of an
    iso-response tuning curve.

    Parameters
    ----------
    freqs : array of test frequencies (Hz) — e.g. the `freqs` returned by
        tuning_curve().
    levels : array of threshold levels (dB SPL) at each frequency, same
        length as `freqs` — e.g. the `thresholds` returned by tuning_curve().
        NaN entries (criterion never reached) are allowed.
    fill_value : how to treat NaN entries. 'zero' (default) treats them as
        fully attenuated (W=0); pass a number (e.g. the upper level bound
        used in the search) to substitute that level instead.

    Returns
    -------
    erb_hz : estimated ERB, in Hz.
    W : the normalized power-gain curve that was integrated (for plotting).
    """
    freqs = np.asarray(freqs, dtype=float)
    levels = np.asarray(levels, dtype=float).copy()

    order = np.argsort(freqs)
    freqs = freqs[order]
    levels = levels[order]

    if fill_value == 'zero':
        levels_filled = np.where(np.isnan(levels), np.inf, levels)
    else:
        levels_filled = np.where(np.isnan(levels), float(fill_value), levels)

    L_min = np.nanmin(levels)             # tip threshold -> W = 1 there
    dL = levels_filled - L_min            # extra level (dB) needed, relative to tip
    # this flips and normalized the tuning curve such that the maximum is now at 0 dB,
    # which simplifies the ERB calculation.

    # TODO: HERE WRITE YOUR CODE TO GET erb_hz (ERB bandwidth in Hz) and W (the normalized power gain curve for plotting)

    $$
    return erb_hz, W

levels_to_test = [20, 40, 60, 80, 100]
erb_results = {}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for L in levels_to_test:
    place_cf, target, tc_freqs, tc_levels = tuning_curve(ref_freq=1000, ref_level_db=L)
    erb_hz, W = estimate_erb(tc_freqs, tc_levels)
    erb_results[L] = erb_hz
    print(f"Level {L:3d} dB SPL -> ERB = {erb_hz:7.1f} Hz  (place CF = {place_cf:.0f} Hz)")

    axes[0].semilogx(tc_freqs, tc_levels, marker='o', ms=3, label=f"{L} dB")
    axes[1].semilogx(tc_freqs, W, marker='o', ms=3, label=f"{L} dB")

axes[0].set_xlabel('Frequency (Hz)'); axes[0].set_ylabel('Threshold (dB SPL)')
axes[0].set_title('Tuning curves'); axes[0].legend(fontsize=8)

axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Normalized power gain W(f)')
axes[1].set_title('Implied filter shape'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- ERB vs. level, with the Glasberg & Moore (1990) human reference -------
erb_ref = erb_cam(1000)
print(f"\nGlasberg & Moore (1990) ERB at 1000 Hz: {erb_ref:.1f} Hz")

plt.figure(figsize=(6, 5))
plt.plot(levels_to_test, [erb_results[L] for L in levels_to_test], marker='o', label='DRNL model (simulated)')
plt.axhline(erb_ref, color='k', linestyle='--', label=f'Glasberg & Moore (1990), {erb_ref:.1f} Hz')
plt.xlabel('Reference level (dB SPL)')
plt.ylabel('Estimated ERB (Hz)')
plt.title('ERB vs. level, from DRNL iso-response tuning curves at 1 kHz')
plt.grid(alpha=0.3)
plt.legend()
plt.show()